# Learned meshless FD with a kernel mixture: Test Problem 1

Discretization: the FD solver of `PHS_fd_TP1.m` (fitted interface nodes, one stencil per row, the `nloc` nearest nodes of the same side).

Weights of one stencil: RBF-FD weights of the kernel $K=\sum_k a_k K_k$, $a_k\ge 0$, $\sum_k a_k=1$, with polynomials of degree `deg`.
The dictionary $K_k$ contains PHS $r^3,-r^5,r^7$, Gaussians and Matérn-5/2 kernels (shape parameters in scaled stencil coordinates).
A small network maps the stencil geometry to $a$; every row is exact on the polynomials for every $a$.

Loss (manufactured solutions on the training node sets, new random solutions at every step):
$$L=\frac{\|E\|^2}{\|E_0\|^2}+\lambda\,\frac{\rho}{\rho_0},\qquad E=-A^{-1}\tau,\quad \rho=\operatorname{mean}_i\sum_j|\hat w_{ij}|\,\|\hat y_j\|^{\mu},\ \mu=\texttt{deg},$$
where $E_0,\rho_0$ belong to the baseline (PHS $r^5$ only). The exact solution of the test problem is used only for testing.

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
import lfd
torch.manual_seed(0)

## Settings and training

In [ ]:
problem      = lfd.TP1
deg          = 4                 # polynomial degree (nloc = 3Q nearest nodes)
train_levels = [2, 3]            # node levels used for training (uniform and Halton)
steps        = 200               # Adam steps
lam          = 0.1               # weight of the growth term (eq. 20, mu = deg)

model = lfd.MixtureFD(deg=deg)
print('kernels:', model.names)
geoms = [lfd.Geometry(problem, m, nt, deg) for m in train_levels for nt in ('uniform', 'halton')]
print('unknowns per training node set:', [g.N for g in geoms])
hist = lfd.train(model, geoms, steps=steps, nsol=8, lam=lam, log=25)
torch.save(model.state_dict(), f'{problem.name}_mixture_deg{deg}.pt')

In [ ]:
it = [h[0] for h in hist]; V = np.array([h[1] for h in hist])
plt.figure(figsize=(6, 3.5))
for j, g in enumerate(geoms):
    plt.semilogy(it, V[:, j], 'o-', label=f'm = {g.m}, {g.ntype}')
plt.axhline(1, color='k', lw=0.5); plt.xlabel('step'); plt.ylabel(r'validation $\|E\|^2/\|E_0\|^2$')
plt.legend(); plt.grid(alpha=0.3); plt.title('Error relative to the baseline (manufactured solutions)'); plt.show()

## What the network learned
Mean mixture coefficients over the Laplacian rows and over the flux rows.

In [ ]:
np.set_printoptions(precision=3, suppress=True, linewidth=150)
print(' ' * 22, '  '.join(f'{n:>7}' for n in model.names))
for nt in ('uniform', 'halton'):
    g = lfd.Geometry(problem, 3, nt, deg)
    lap, dn = model.mixture_map(g)
    print(f'{nt:8s} Laplacian rows', '  '.join(f'{v:7.3f}' for v in lap.mean(0)))
    print(f'{nt:8s} flux rows     ', '  '.join(f'{v:7.3f}' for v in dn.mean(0)))

## Test Problem 1: baseline (PHS $r^5$) against the learned mixture
Errors over the interior nodes; `sigma` estimates $\|A^{-1}\|_\infty$.

In [ ]:
levels = range(0, 5)
res = {nt: lfd.table(model, problem, levels, nt) for nt in ('uniform', 'halton')}

## Reference: each dictionary kernel alone
Classical RBF-FD with one kernel (same stencils, same polynomial degree).

In [ ]:
single = {nt: lfd.single_kernel_table(model, problem, levels, nt) for nt in ('uniform', 'halton')}

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, nt in zip(ax, ('uniform', 'halton')):
    N = [r[2]['Nint'] for r in res[nt]]
    a.loglog(N, [r[2]['rms'] for r in res[nt]], 'o-', label='baseline (PHS $r^5$)')
    a.loglog(N, [r[3]['rms'] for r in res[nt]], 's-', label='learned mixture')
    a.loglog(N, single[nt]['phs7'], 'k:', label='PHS $r^7$ alone')
    a.set_xlabel('$N_{int}$'); a.set_ylabel('rms error'); a.set_title(f'{problem.name}, {nt} nodes, degree {deg}')
    a.grid(alpha=0.3); a.legend()
plt.tight_layout(); plt.show()